## **For this notebook to work, you must execute ALL the cells in the order in which they appear.**

In [ ]:

# Link your Google Drive so that Colab can access it.
# Colab will ask for authorization to access your Google Drive.
from google.colab import drive
drive.mount("/content/drive")

# 3. Create a Spark session with a custom name.
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("Colab_Session_Name") \
.getOrCreate()


Mounted at /content/drive


# **Upload a CSV that does not have column headers**

In [ ]:
# read a CSV file from Drive and display it
df = spark.read.format("csv").option("header","true").load("/content/drive/MyDrive/2019.csv")
df.show()    # In Fabric, use display(df)
# Notice that there are no column headers, and the first row of data takes their place.

+-------+---+----------+------------------+-----------------------------+-----------------------+---+--------+--------+
|SO43701| 11|2019-07-01|       Christy Zhu|christy12@adventure-works.com|Mountain-100 Silver, 44| 16| 3399.99|271.9992|
+-------+---+----------+------------------+-----------------------------+-----------------------+---+--------+--------+
|SO43704|  1|2019-07-01|        Julio Ruiz|         julio1@adventure-...|   Mountain-100 Blac...|  1| 3374.99|269.9992|
|SO43705|  1|2019-07-01|         Curtis Lu|         curtis9@adventure...|   Mountain-100 Silv...|  1| 3399.99|271.9992|
|SO43700|  1|2019-07-01|      Ruben Prasad|         ruben10@adventure...|     Road-650 Black, 62|  1|699.0982| 55.9279|
|SO43703|  1|2019-07-01|    Albert Alvarez|         albert7@adventure...|       Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43697|  1|2019-07-01|       Cole Watson|         cole1@adventure-w...|       Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43699|  1|2019-07-01|     Sydney Wrig

# **option("header","false") will indicate that the CSV does not contain column headers**

In [ ]:
# Tell Spark that the CSV does not have column headers. It will temporarily replace them with C0, C1, C2, etc.
df = spark.read.format("csv").option("header","false").load("/content/drive/MyDrive/2019.csv")
df.show()     # en Fabric use display(df)

+-------+---+----------+-----------------+--------------------+--------------------+---+--------+--------+
|    _c0|_c1|       _c2|              _c3|                 _c4|                 _c5|_c6|     _c7|     _c8|
+-------+---+----------+-----------------+--------------------+--------------------+---+--------+--------+
|SO43701|  1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|  1| 3399.99|271.9992|
|SO43704|  1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|  1| 3374.99|269.9992|
|SO43705|  1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|  1| 3399.99|271.9992|
|SO43700|  1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|  1|699.0982| 55.9279|
|SO43703|  1|2019-07-01|   Albert Alvarez|albert7@adventure...|    Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43697|  1|2019-07-01|      Cole Watson|cole1@adventure-w...|    Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43699|  1|2019-07-01|    Sydney Wr

## **Assign column headers to the DataFrame**

In [ ]:
# assign column headers to the DataFrame
from pyspark.sql.types import *

orderSchema = StructType([
   StructField("SalesOrderNumber", StringType()),
   StructField("SalesOrderLineNumber", IntegerType()),
   StructField("OrderDate", DateType()),
   StructField("CustomerName", StringType()),
   StructField("Email", StringType()),
   StructField("Item", StringType()),
   StructField("Quantity", IntegerType()),
   StructField("UnitPrice", FloatType()),
   StructField("Tax", FloatType())
])

df = spark.read.format("csv").schema(orderSchema).load("/content/drive/MyDrive/2019.csv")
df.show()     # In Fabric, use display(df)

+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|         

In [ ]:
# 1. Convert to Pandas and save in Colab (this file is lost when the Colab session ends)
df_pandas = df.toPandas()
df_pandas.to_csv("archivo_exportado.csv", index=False)

# 2. Download the CSV file to your computer
from google.colab import files
files.download("archivo_exportado.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Save the files and retrieve them when needed.**

In [ ]:
# Save the CSV as Parquet in Google Drive, in the "Mis_Parquet" folder.

df.write.format("parquet").mode("overwrite").save(
    "/content/drive/MyDrive/Mis_Parquet/orders_parquet"
)


In [ ]:
# Read it back at any time (Spark automatically detects the schema)
df = spark.read.format("parquet").load(
    "/content/drive/MyDrive/Mis_Parquet/orders_parquet"
)
df.show()

+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|         